In [1]:
!pip install -q gradio openai-whisper
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.8 MB/s eta 0:00:00


In [2]:
import whisper
import gradio as gr
import os

# Load Whisper ASR model
model = whisper.load_model("base")

print("✅ ASR Model loaded successfully!")
print("🎙️ Ready to convert speech into text.")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 146MiB/s]


✅ ASR Model loaded successfully!
🎙️ Ready to convert speech into text.


In [3]:
def transcribe_audio(audio_file):
    if audio_file is None:
        return "⚠️ Please upload or record an audio file."

    try:
        # Transcribe speech
        result = model.transcribe(audio_file)

        text = result["text"].strip()

        if not text:
            return "⚠️ No speech detected in the audio."

        return text

    except Exception as e:
        return f"❌ Error during transcription: {str(e)}"


print("✅ Speech-to-text function is ready!")

✅ Speech-to-text function is ready!


In [5]:
import gradio as gr

custom_css = """
body {
    background: #0b1020 !important;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    background: #0b1020 !important;
    font-family: Arial, sans-serif !important;
}

.hero {
    text-align: center;
    padding: 35px 20px 25px;
}

.hero h1 {
    font-size: 42px !important;
    margin-bottom: 8px !important;
    color: white !important;
}

.hero p {
    color: #aeb8d0 !important;
    font-size: 16px !important;
}

.card {
    background: #151c32 !important;
    border: 1px solid #273252 !important;
    border-radius: 18px !important;
    padding: 22px !important;
}

.section-title {
    color: white !important;
    font-size: 20px !important;
    font-weight: 700 !important;
    margin-bottom: 12px !important;
}

.info-box {
    background: #11182b;
    border: 1px solid #273252;
    border-radius: 14px;
    padding: 15px;
    color: #b9c3da;
    text-align: center;
}

button {
    border-radius: 12px !important;
    font-weight: 600 !important;
}

#convert-btn {
    height: 52px !important;
    font-size: 16px !important;
}

.footer {
    text-align: center;
    color: #727d98;
    padding: 25px;
    font-size: 13px;
}
"""

def word_count(text):
    if not text:
        return "0 words"
    return f"{len(text.split())} words"

def char_count(text):
    if not text:
        return "0 characters"
    return f"{len(text)} characters"

def transcribe_and_stats(audio_file):
    if audio_file is None:
        return "⚠️ Please record or upload an audio file.", "0 words", "0 characters"

    try:
        result = model.transcribe(audio_file)
        text = result["text"].strip()

        if not text:
            return "⚠️ No speech detected.", "0 words", "0 characters"

        return text, word_count(text), char_count(text)

    except Exception as e:
        return f"❌ Error: {str(e)}", "0 words", "0 characters"


with gr.Blocks(
    title="ASR Studio | Speech Recognition",
    css=custom_css,
    theme=gr.themes.Base()
) as app:

    # Hero Section
    gr.HTML("""
    <div class="hero">
        <h1>🎙️ ASR Studio</h1>
        <p>
            Automatic Speech Recognition powered by AI
            <br>
            Transform spoken words into clear, editable text.
        </p>
    </div>
    """)

    with gr.Row():

        # LEFT PANEL
        with gr.Column(scale=1, elem_classes="card"):

            gr.HTML("""
            <div class="section-title">
                🎤 Audio Input
            </div>
            """)

            audio_input = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="Record or Upload Audio"
            )

            gr.Markdown(
                "Supported audio: WAV, MP3, M4A and other common formats."
            )

            convert_btn = gr.Button(
                "✨ Convert Speech to Text",
                variant="primary",
                elem_id="convert-btn"
            )

            clear_btn = gr.Button(
                "🗑️ Clear",
                variant="secondary"
            )

        # RIGHT PANEL
        with gr.Column(scale=1, elem_classes="card"):

            gr.HTML("""
            <div class="section-title">
                📝 Transcription
            </div>
            """)

            text_output = gr.Textbox(
                label="Recognized Text",
                placeholder="Your transcribed speech will appear here...",
                lines=12
            )

            with gr.Row():
                words = gr.Textbox(
                    label="Words",
                    value="0 words",
                    interactive=False
                )

                characters = gr.Textbox(
                    label="Characters",
                    value="0 characters",
                    interactive=False
                )

    # Information section
    with gr.Row():

        gr.HTML("""
        <div class="info-box">
            🎯 <b>Accurate</b><br>
            AI-powered transcription
        </div>
        """)

        gr.HTML("""
        <div class="info-box">
            ⚡ <b>Fast</b><br>
            Quick speech processing
        </div>
        """)

        gr.HTML("""
        <div class="info-box">
            🔒 <b>Simple</b><br>
            Easy-to-use interface
        </div>
        """)

    gr.HTML("""
    <div class="footer">
        ASR Studio • Automatic Speech Recognition System
        <br>
        Built with Python, Whisper AI & Gradio
    </div>
    """)

    # Convert
    convert_btn.click(
        fn=transcribe_and_stats,
        inputs=audio_input,
        outputs=[text_output, words, characters]
    )

    # Clear
    clear_btn.click(
        fn=lambda: (None, "", "0 words", "0 characters"),
        inputs=None,
        outputs=[audio_input, text_output, words, characters]
    )

app.launch(share=True)

/tmp/ipykernel_1265/3096312762.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6d7cadf7d70445eaa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
